# NYT Sushi Mentions

Borrowing code from Lara


In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import re

need to create an account and an "app" project to get API key

do this here: https://developer.nytimes.com/apis 

In [ ]:
BASE_URL = "https://api.nytimes.com/svc/search/v2/articlesearch.json"

HEADLINE_PHRASE = "space race"
HEADLINE_RE = re.compile(rf"\b{re.escape(HEADLINE_PHRASE)}\b", re.IGNORECASE)

def headline_has_phrase(doc):
    headline = doc.get("headline") or {}
    main = headline.get("main") or ""
    default = headline.get("default") or ""
    print_h = headline.get("print_headline") or ""
    return bool(
        HEADLINE_RE.search(main)
        or HEADLINE_RE.search(default)
        or HEADLINE_RE.search(print_h)
    )

def fetch_space_race():
    results = []

    for year in range(1980, 2020):
        print(f"\nChecking year {year}...")

        for page in range(100):
            params = {
                "q": '"space race"',
                "begin_date": f"{year}0101",
                "end_date": f"{year}1231",
                "page": page,
                "sort": "oldest",
                "api-key": API_KEY,
            }

            res = requests.get(BASE_URL, params=params, timeout=30)
            data = res.json()

            response = data.get("response", {}) or {}
            meta = response.get("meta", {}) or {}
            docs = response.get("docs", []) or []

            print(f"  page {page}: docs={len(docs)} meta_hits={meta.get('hits')}")

            if not docs:
                break

            for doc in docs:
                if headline_has_phrase(doc):
                    row = {
                        "year": year,
                        "pub_date": doc.get("pub_date"),
                        "headline_main": (doc.get("headline") or {}).get("main"),
                        "headline_default": (doc.get("headline") or {}).get("default"),
                        "headline_print": (doc.get("headline") or {}).get("print_headline"),
                        "snippet": doc.get("snippet"),
                        "lead_paragraph": doc.get("lead_paragraph"),
                        "web_url": doc.get("web_url"),
                        "uri": doc.get("uri"),
                    }
                    results.append(row)
                    print("MATCH:", row["headline_main"] or row["headline_default"] or row["headline_print"])

    df = pd.DataFrame(results)
    df.to_csv("space_race_results.csv", index=False)

    print("\nDone.")
    print(f"Total matches found: {len(df)}")
    display(df)

    return df

df_space_race = fetch_space_race()